In [1]:
import pandas as pd
import numpy as np

In [2]:
MODEL_FILES = {
    "LightGBM": "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",
    "RF": "Prescriptive/RF_prescriptive_optimizer_inputs.csv",
    "XGBoost": "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv",
}

In [3]:
required_cols = [
    "LOG_ID",
    "DURATION_P10_MINS",
    "DURATION_P50_MINS",
    "DURATION_P90_MINS",
]

for model_name, path in MODEL_FILES.items():

    df = pd.read_csv(path)

    assert len(df) == 17083
    assert list(df.columns) == required_cols
    assert df["LOG_ID"].is_unique
    assert not df[required_cols].isna().any().any()

    assert np.all(
        df["DURATION_P10_MINS"]
        <= df["DURATION_P50_MINS"]
    )

    assert np.all(
        df["DURATION_P50_MINS"]
        <= df["DURATION_P90_MINS"]
    )

    print(model_name, "PASS", df.shape)

LightGBM PASS (17083, 4)
RF PASS (17083, 4)
XGBoost PASS (17083, 4)


In [4]:
lgb = pd.read_csv(MODEL_FILES["LightGBM"])
rf = pd.read_csv(MODEL_FILES["RF"])
xgb = pd.read_csv(MODEL_FILES["XGBoost"])

assert lgb["LOG_ID"].tolist() == rf["LOG_ID"].tolist()
assert lgb["LOG_ID"].tolist() == xgb["LOG_ID"].tolist()

print("Cross-model LOG_ID alignment: PASS")

Cross-model LOG_ID alignment: PASS


In [5]:
cohort = pd.read_csv("Cohort/fixed_cohort.csv")

for model_name, path in MODEL_FILES.items():
    pred = pd.read_csv(path)

    missing_ids = (
        set(cohort["LOG_ID"])
        - set(pred["LOG_ID"])
    )

    assert not missing_ids, (
        f"{model_name}: fixed cohort IDs missing: {missing_ids}"
    )

print("Fixed 12-case cohort compatibility: PASS")

Fixed 12-case cohort compatibility: PASS


In [6]:
lambdas = np.round(
    np.arange(0.0, 1.01, 0.1),
    1
)

cap_rows = []

for model_name, path in MODEL_FILES.items():

    df = pd.read_csv(path)

    p50 = df["DURATION_P50_MINS"].to_numpy()
    p90 = df["DURATION_P90_MINS"].to_numpy()

    for lam in lambdas:

        uncapped = (
            p50
            + lam * (p90 - p50)
        )

        affected = uncapped > 360

        cap_rows.append({
            "Model": model_name,
            "Lambda": lam,
            "N": len(df),
            "N_Capped": int(affected.sum()),
            "Pct_Capped": float(
                affected.mean() * 100
            ),
            "Max_Uncapped": float(
                uncapped.max()
            )
        })

cap_audit = pd.DataFrame(cap_rows)

print(cap_audit)

       Model  Lambda      N  N_Capped  Pct_Capped  Max_Uncapped
0   LightGBM     0.0  17083       237    1.387344    516.391135
1   LightGBM     0.1  17083       285    1.668325    540.300184
2   LightGBM     0.2  17083       395    2.312240    564.209234
3   LightGBM     0.3  17083       475    2.780542    588.118283
4   LightGBM     0.4  17083       584    3.418603    612.062558
5   LightGBM     0.5  17083       797    4.665457    639.031049
6   LightGBM     0.6  17083      1042    6.099631    692.267530
7   LightGBM     0.7  17083      1293    7.568928    778.516209
8   LightGBM     0.8  17083      1657    9.699701    864.764889
9   LightGBM     0.9  17083      2068   12.105602    951.013568
10  LightGBM     1.0  17083      4950   28.976175   1037.262247
11        RF     0.0  17083         0    0.000000    176.603005
12        RF     0.1  17083         0    0.000000    228.116617
13        RF     0.2  17083         0    0.000000    284.088852
14        RF     0.3  17083         0   

In [7]:
cap_audit.to_csv(
    "Results/duration_cap_audit.csv",
    index=False
)